# Частина 2. Аналіз споживання електроенергії

Датасет **Individual household electric power consumption** містить хвилинні вимірювання споживання електроенергії одного домогосподарства у Франції протягом ~4 років (2006–2010).  

Основні стовпці:  
| Стовпець | Опис |
|---|---|
| `Global_active_power` | Загальна активна потужність (кВт) |
| `Global_reactive_power` | Реактивна потужність (кВт) |
| `Voltage` | Напруга (В) |
| `Global_intensity` | Сила струму (А) |
| `Sub_metering_1` | Кухня: бойлер, кондиціонер (Вт·год) |
| `Sub_metering_2` | Пральна/сушарка/холодильник (Вт·год) |
| `Sub_metering_3` | Водонагрівач та кондиціонер (Вт·год) |

In [1]:
import pandas as pd
import timeit
import numpy as np

## 1. Завантаження датасету

Файл розділений крапкою з комою (`;`), дата і час об'єднуються в один стовпець `DateTime`.  
Пропущені значення позначені символом `?` - вони автоматично замінюються на `NaN`.

In [ ]:
import requests
import zipfile
import os

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/household_power_consumption.txt"):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
    print("Завантаження датасету...")
    
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    
    total = int(response.headers.get("content-length", 0))
    downloaded = 0
    
    with open("data/dataset.zip", "wb") as f:
        for chunk in response.iter_content(chunk_size=65536):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"\r  {pct:.1f}%  ({downloaded // 1024 // 1024} / {total // 1024 // 1024} MB)", end="", flush=True)
    
    print("\nРозпаковка...")
    with zipfile.ZipFile("data/dataset.zip", "r") as z:
        z.extractall("data/")
    os.remove("data/dataset.zip")
    print("Готово.")
else:
    print("Датасет вже існує, пропускаємо завантаження.")

df = pd.read_csv(
    "data/household_power_consumption.txt",
    sep=";",
    na_values=["?"],
    low_memory=False
)

df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df = df.drop(columns=["Date", "Time"])

Датасет вже існує, пропускаємо завантаження.


In [3]:
df.shape

(2075259, 8)

In [4]:
df.dtypes

Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
DateTime                 datetime64[us]
dtype: object

In [5]:
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


## 2. Очищення даних

Видаляємо рядки з пропущеними значеннями (`NaN`). Виводимо кількість пропусків **до** видалення, щоб оцінити масштаб очищення.

In [6]:
def clean_power_data(df: pd.DataFrame) -> pd.DataFrame:
    df.dropna(inplace=True)
    return df

print(f"Пропуски до очищення: {df.isnull().sum().sum()}")
df = clean_power_data(df)
print(f"Пропуски після очищення: {df.isnull().sum().sum()}")

Пропуски до очищення: 181853
Пропуски після очищення: 0


## 3. Фільтрація: потужність > 5 кВт

Відбираємо хвилини, коли сумарна активна потужність перевищувала **5 кВт** - пікове навантаження.  
Час виконання замірюється 10 разів за допомогою `timeit` для оцінки ефективності.

In [ ]:
def power_above_5kw(df: pd.DataFrame) -> pd.DataFrame:
    return df[df['Global_active_power'] > 5]

t = timeit.timeit(lambda: power_above_5kw(df), number=10)
res4 = power_above_5kw(df)
print(f"Записів з потужністю > 5 кВт: {len(res4)}")
print(f"Час виконання (10 ітерацій): {t:.4f} с")
res4.head()

Записів з потужністю > 5 кВт: 17547
Час виконання (10 ітерацій): 0.0164 с
    Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
1                 5.360                  0.436   233.63              23.0   
2                 5.374                  0.498   233.29              23.0   
3                 5.388                  0.502   233.74              23.0   
11                5.412                  0.470   232.78              23.2   
12                5.224                  0.478   232.99              22.4   

    Sub_metering_1  Sub_metering_2  Sub_metering_3            DateTime  
1              0.0             1.0            16.0 2006-12-16 17:25:00  
2              0.0             2.0            17.0 2006-12-16 17:26:00  
3              0.0             1.0            17.0 2006-12-16 17:27:00  
11             0.0             1.0            17.0 2006-12-16 17:35:00  
12             0.0             1.0            16.0 2006-12-16 17:36:00  


## 4. Фільтрація за силою струму та порівняння груп

Вибираємо записи, де сила струму знаходиться в діапазоні **19–20 А** (близько до максимального навантаження на автомат 20 А).  
Далі залишаємо лише ті рядки, де споживання **Sub_metering_2** (пральна/холодильник) перевищує **Sub_metering_1** (кухня) - тобто «побутова» група споживала більше за «кухню».

In [ ]:
def current_filter(df: pd.DataFrame) -> pd.DataFrame:
    mask_current = df['Global_intensity'].between(19, 20)
    filtered = df[mask_current]
    # Sub_metering_1: кухня (бойлер, кондиціонер)
    # Sub_metering_2: пральна машина, сушарка, холодильник
    mask_appliance = filtered['Sub_metering_2'] > filtered['Sub_metering_1']
    return filtered[mask_appliance]

t = timeit.timeit(lambda: current_filter(df), number=10)
res5 = current_filter(df)
print(f"Результатів: {len(res5)}")
print(f"Час: {t:.4f} с")
res5.head()

Результатів: 2577
Час: 0.0310 с
    Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
10                4.448                  0.498   232.86              19.6   
45                4.464                  0.136   234.66              19.0   
52                4.524                  0.076   234.20              19.6   
54                4.472                  0.000   233.29              19.2   
72                4.536                  0.000   233.54              19.4   

    Sub_metering_1  Sub_metering_2  Sub_metering_3            DateTime  
10             0.0             1.0            17.0 2006-12-16 17:34:00  
45             0.0            37.0            16.0 2006-12-16 18:09:00  
52             0.0             9.0            17.0 2006-12-16 18:16:00  
54             0.0             1.0            16.0 2006-12-16 18:18:00  
72             0.0             1.0            17.0 2006-12-16 18:36:00  


## 5. Середні по вибірці 500 000 записів

Щоб уникнути зміщення через часові патерни (час доби, день тижня), беремо **випадкову** вибірку 500 000 записів (`random_state=42` для відтворюваності) та рахуємо середнє споживання по кожній із трьох груп лічильників.

Це дозволяє оцінити, яка група приладів загалом споживає більше.

In [ ]:
def random_sample_means(df: pd.DataFrame) -> pd.Series:
    sample = df.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

t = timeit.timeit(lambda: random_sample_means(df), number=5)
res6 = random_sample_means(df)
print("Середні по групах споживання (вибірка 500 000):")
print(f"Час (5 ітерацій): {t:.4f} с")
res6

Середні по групах споживання (вибірка 500 000):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64
Час (5 ітерацій): 0.3748 с


## 6. Вечірнє пікове споживання

Аналізуємо **вечірні** години (≥18:00) з потужністю > 6 кВт, де група **Sub_metering_2** є найбільшою серед трьох.  
Датасет розбивається на дві половини:
- Перша половина - беремо **кожен 3-й** запис.
- Друга половина - беремо **кожен 4-й** запис.

Таке розріджування знижує обсяг даних, зберігаючи рівномірний часовий розподіл.

In [ ]:
def evening_high_consumption(df: pd.DataFrame) -> pd.DataFrame:
    mask_time = df['DateTime'].dt.hour >= 18
    mask_power = df['Global_active_power'] > 6
    filtered = df[mask_time & mask_power]
    
    # Група 2 найбільша серед трьох
    mask_g2 = (
        (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) &
        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    )
    filtered = filtered[mask_g2].reset_index(drop=True)
    
    mid = len(filtered) // 2
    first_half  = filtered.iloc[:mid].iloc[::3]   # кожен 3-й
    second_half = filtered.iloc[mid:].iloc[::4]   # кожен 4-й
    
    return pd.concat([first_half, second_half])

t = timeit.timeit(lambda: evening_high_consumption(df), number=5)
res7 = evening_high_consumption(df)
print(f"Результатів: {len(res7)}")
print(f"Час: {t:.4f} с")
res7.head()

Результатів: 310
Час: 0.1479 с
    Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
0                 6.052                  0.192   232.93              26.2   
3                 6.308                  0.116   232.25              27.0   
6                 6.386                  0.374   236.63              27.0   
9                 8.088                  0.262   235.50              34.4   
12                7.230                  0.152   235.22              30.6   

    Sub_metering_1  Sub_metering_2  Sub_metering_3            DateTime  
0              0.0            37.0            17.0 2006-12-16 18:05:00  
3              0.0            36.0            17.0 2006-12-16 18:08:00  
6              1.0            36.0            17.0 2006-12-28 20:58:00  
9              1.0            72.0            17.0 2006-12-28 21:02:00  
12             1.0            73.0            17.0 2006-12-28 21:05:00  


## 7. Нормалізація та стандартизація

Масштабування числових ознак необхідне для алгоритмів ML, чутливих до діапазону значень (наприклад, KNN, SVM, нейронні мережі).

| Метод | Формула | Діапазон | Коли використовувати |
|---|---|---|---|
| **Min-Max** (нормалізація) | `(x - min) / (max - min)` | [0, 1] | Коли потрібен фіксований діапазон |
| **Z-score** (стандартизація) | `(x - mean) / std` | [-∞, +∞] | Коли припускається нормальний розподіл |

In [11]:
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

# Нормалізація (Min-Max)
def normalize_minmax(df, cols):
    df_n = df.copy()
    for col in cols:
        mn, mx = df[col].min(), df[col].max()
        df_n[col] = (df[col] - mn) / (mx - mn)
    return df_n

t_norm = timeit.timeit(lambda: normalize_minmax(df, numeric_cols), number=5)
df_normalized = normalize_minmax(df, numeric_cols)
print(f"Час нормалізації (5 ітерацій): {t_norm:.4f} с")

# Стандартизація (Z-score)
def standardize_zscore(df, cols):
    df_s = df.copy()
    for col in cols:
        df_s[col] = (df[col] - df[col].mean()) / df[col].std()
    return df_s

t_std = timeit.timeit(lambda: standardize_zscore(df, numeric_cols), number=5)
df_standardized = standardize_zscore(df, numeric_cols)
print(f"Час стандартизації (5 ітерацій): {t_std:.4f} с")
print("\nНормалізований датасет (перші рядки):")
df_normalized[numeric_cols].head()

Час нормалізації (5 ітерацій): 0.5024 с
Час стандартизації (5 ітерацій): 0.9502 с

Нормалізований датасет (перші рядки):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


## 8. Кореляційний аналіз

Вимірюємо зв'язок між **загальною активною потужністю** і **напругою**:

- **Коефіцієнт Пірсона** - лінійна кореляція. Значення від -1 (повна обернена) до +1 (повна пряма). Чутливий до викидів.
- **Коефіцієнт Спірмена** - ранговa кореляція (монотонна залежність). Стійкий до викидів та нелінійних зв'язків.

Якщо обидва коефіцієнти схожі → зв'язок лінійний. Якщо Спірмен >> Пірсон → нелінійний монотонний зв'язок.

In [12]:
col1, col2 = 'Global_active_power', 'Voltage'

def calc_pearson(df, c1, c2):
    return df[c1].corr(df[c2], method='pearson')

def calc_spearman(df, c1, c2):
    return df[c1].corr(df[c2], method='spearman')

t_p = timeit.timeit(lambda: calc_pearson(df, col1, col2), number=5)
t_s = timeit.timeit(lambda: calc_spearman(df, col1, col2), number=5)

pearson  = calc_pearson(df, col1, col2)
spearman = calc_spearman(df, col1, col2)

print(f"Коефіцієнт Пірсона  ({col1} vs {col2}): {pearson:.4f}  | Час (5 ітерацій): {t_p:.4f} с")
print(f"Коефіцієнт Спірмена ({col1} vs {col2}): {spearman:.4f}  | Час (5 ітерацій): {t_s:.4f} с")

Коефіцієнт Пірсона  (Global_active_power vs Voltage): -0.3998  | Час (5 ітерацій): 0.0738 с
Коефіцієнт Спірмена (Global_active_power vs Voltage): -0.3252  | Час (5 ітерацій): 2.2664 с


## 9. One Hot Encoding (OHE) категоріального атрибуту

Категоріальний атрибут **«частина доби»** (`time_of_day`) не можна передавати в ML-моделі у вигляді рядка.  
Розбиваємо на чотири бінарні стовпці:
- `tod_morning` (6:00–12:00)
- `tod_afternoon` (12:00–18:00)
- `tod_evening` (18:00–22:00)
- `tod_night` (22:00–6:00)

В кожному рядку рівно одна з цих колонок = `True`, решта - `False`.

In [13]:
# Додаємо категоріальний атрибут - частина доби
def time_of_day(hour):
    if 6 <= hour < 12:    return "morning"
    elif 12 <= hour < 18: return "afternoon"
    elif 18 <= hour < 22: return "evening"
    else:                 return "night"

df['time_of_day'] = df['DateTime'].dt.hour.apply(time_of_day)

df_ohe = pd.get_dummies(df, columns=['time_of_day'], prefix='tod')
print("Стовпці після One Hot Encoding:")
print([c for c in df_ohe.columns if c.startswith('tod')])
df_ohe.filter(like='tod').head()

Стовпці після One Hot Encoding:
['tod_afternoon', 'tod_evening', 'tod_morning', 'tod_night']
   tod_afternoon  tod_evening  tod_morning  tod_night
0           True        False        False      False
1           True        False        False      False
2           True        False        False      False
3           True        False        False      False
4           True        False        False      False
